# PAPER-EXP-05A v2 — Numerical-Stability Repair

This is an **infrastructure-only repair** of PAPER-EXP-05A.

The frozen scientific system is unchanged:

- 3× EfficientNet-B0 @ 300 px
- 3× EfficientNet-B3 @ 300 px
- fixed TTA7
- equal 1/3 averaging within family
- fixed 50/50 B0:B3 fusion

The original run encountered a non-finite CUDA FP16 probability for
`b3_seed_20260821 | PAD_UFES_20`.

v2 keeps FP16 as the primary inference path. **Only a row that becomes non-finite**
is recomputed with the identical frozen model and identical TTA7 views in FP32.
No labels are used for this fallback.

The notebook also safely resumes a failed run: an existing prediction CSV is skipped
only after exact row/key/model/finite/probability-sum validation.


In [1]:
from pathlib import Path

SCRIPT = 'from __future__ import annotations\n\nimport argparse\nimport base64\nimport contextlib\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport sys\nimport time\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom PIL import Image, ImageOps\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_recall_fscore_support,\n    recall_score,\n    roc_auc_score,\n    roc_curve,\n)\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom torchvision import models, transforms\nfrom tqdm.auto import tqdm\n\nCODE_VERSION = "PAPEREXP05A_FROZEN_PAPEREXP01C_ALL_EXTERNAL_2026-08-25_v2_NUMERICAL_FALLBACK"\nINPUT_ROOT = Path("/kaggle/input")\nWORKING_ROOT = Path("/kaggle/working")\nOUTPUT_ROOT = WORKING_ROOT / "PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal"\nPRED_ROOT = OUTPUT_ROOT / "model_predictions"\nFINAL_ROOT = OUTPUT_ROOT / "final_results"\n\nCLASS_NAMES: Tuple[str, ...] = ("MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC")\nCLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}\nNUM_CLASSES = len(CLASS_NAMES)\nIMAGE_SIZE = 300\nMEL_INDEX = CLASS_TO_INDEX["MEL"]\n\nTTA_VIEWS = (\n    "original",\n    "horizontal",\n    "vertical",\n    "horizontal_vertical",\n    "rotation90",\n    "rotation180",\n    "rotation270",\n)\n\nCHECKPOINT_HASHES = {\'b0_seed_20260820\': \'30efcb4f88cfddd4f6ece227d920fbab27caf38e821f2e442941510b199bfb39\', \'b0_seed_20260821\': \'a79eef838354a6613d75b7aa455d643ccd0f3309b23a31f177c3169f723575cb\', \'b0_seed_20260822\': \'f258083d9bb36858ad96ce2d2fd8ba54a0834dc5b8f38dcbb1ee762783509587\', \'b3_seed_20260820\': \'59a23d41c48cb1ccbf9184e9cf67c115717775560283bb4a1da95f14698dc296\', \'b3_seed_20260821\': \'723176d2c08b759b9a8c2cbbfe245840ad844f1a77da748948c75c09afc51d69\', \'b3_seed_20260822\': \'2dc2cf4d28b6f5b8ccdb61a173aff25ba80ab2e0c794dfdfafcb104a87d89a4d\'}\n\nB0_MODEL_NAMES = (\'b0_seed_20260820\', \'b0_seed_20260821\', \'b0_seed_20260822\')\nB3_MODEL_NAMES = (\'b3_seed_20260820\', \'b3_seed_20260821\', \'b3_seed_20260822\')\nALL_MODEL_NAMES = B0_MODEL_NAMES + B3_MODEL_NAMES\n\nMODEL_CONFIG = {\n    "b0_seed_20260820": {"batch_size": 64, "family": "B0"},\n    "b0_seed_20260821": {"batch_size": 64, "family": "B0"},\n    "b0_seed_20260822": {"batch_size": 64, "family": "B0"},\n    "b3_seed_20260820": {"batch_size": 40, "family": "B3"},\n    "b3_seed_20260821": {"batch_size": 40, "family": "B3"},\n    "b3_seed_20260822": {"batch_size": 40, "family": "B3"},\n}\n\nCLEAN_INTERNAL_REFERENCE = {\'experiment\': \'PAPER-EXP-01C\', \'scientific_status\': \'final frozen lesion-safe clean internal benchmark\', \'test_images\': 971, \'test_lesions\': 749, \'test_manifest_sha256\': \'5c6f3cdca8ae60e200d6e5f5f1d619b3f08df71a2dadab95cdad7948a2834eec\', \'accuracy\': 0.8682, \'macro_f1\': 0.7333, \'balanced_accuracy\': 0.7076, \'melanoma_recall\': 0.7524, \'macro_auc\': 0.9756}\n\nDATASET_CONFIG = {\n    "DERM12345": {\n        "tokens": ("derm12345", "derm-12345"),\n        "expected_n": 2321,\n        "supported_classes": ("MEL", "NV", "BCC", "AKIEC", "BKL"),\n        "bootstrap_seed": 82001,\n    },\n    "BCN20000": {\n        "tokens": ("bcn20000", "bcn-20000"),\n        "expected_n": 11747,\n        "supported_classes": ("MEL", "NV", "BCC", "AKIEC", "BKL"),\n        "bootstrap_seed": 82002,\n    },\n    "Derm7pt": {\n        "tokens": ("derm7pt", "derm-7pt", "7pt"),\n        "expected_n": 355,\n        "supported_classes": ("MEL", "NV", "BCC", "BKL"),\n        "bootstrap_seed": 82003,\n    },\n    "PAD_UFES_20": {\n        "tokens": ("pad-ufes", "pad_ufes", "padufes"),\n        "expected_n": 2092,\n        "supported_classes": ("MEL", "NV", "BCC", "AKIEC", "BKL"),\n        "bootstrap_seed": 82004,\n    },\n}\n\nMANIFEST_PAYLOADS = {"DERM12345": "...", "BCN20000": "...", "Derm7pt": "H4sIACVdhWoC/12ay25kNw6G9/MsQqz7Zdnd6fJcHGNWs22UL6faQbnSSGYQIE8/vyhKFL0wwPQX6UgUSZFUvb2fL6/fvr/d/mveSHx7+Xb77fd3c/n9t//9wH+Z9/OPH68v356v5z/++Nvj14e7x9erDemnX39czI1EE83nL18Wi26x6ExRrJbFajFNsSbjmjPOaVgFVuPSDp21E0I0Lg/4C+A7/sUPSKLxY9pPL9c7/IXSCJ5JNH5s5P6v693lrysPJMn4sdb78/UOf3M5FxJN8ARPnx7uTv1fMsGDRBPCgD8DvlxtGWs9SDRhbOT+gmkv+A5PS6IJYyMnQPzZOuBBogljQafvgN+vNvG0JJroNPQCvYljtY8n6OfAGbHySDQxmMf/CKvCqomps6k6n5bqfDKJxn16BXu9xjg+eCbRpKhYFBZNSoplYdmkvLHk7GQQTaqd8Un5OE8KJ5jtRuI6Q6wj+05OX3AQz6JOEk2mdcxzsHGdg8WMhXSCA3zEqVk2fRJNccQ+gz3hRMf6bySa4ndWhVUw0skj1vKIBVjWM4kGlrixYBcL1lQ1jnV5I9HUrJjMibOrZWcpLIaDa1axLCyb5naWZVzGOL+zKqyCqe/VJCyZVnd9Or/06bzpDrxB7xaE5zobFJSRvo+MClaBCBhWnWEIC8I9nW0KNoENccjvcHoLiYDKXYJf7oKQ4NxY0D1gd2j+JomAeYdRYOyw7HBaFYnG+XEkfweEQztWLYmAY9p/AL7BmHhBJAIWBatAaMi3Hc4AT6JxISlYBBZA9c25lbexlaC+GeWbMEoXxjcfAK9YP++TRONiVLAKxMgRiyb0dkGPU4l5QvoXLxBKiEVB+abv36w7DDKyn2ey+zejwNih32GRafuRJbXPFhdsEdeVU1D22S+6EbP4LltH9j6OLNcdurig69O2HQaBAbCoaad7kgg4YrlH3H26OvbPM4mAVcEsEPsckWvBKhCrrXaH7A5nEgG9gkUgDKyOBQXA56vn4HUm0bgRoT45wPOKiGcSAbOCWSBWO2LUhEkg4p4fcWjC7BfEiXirvpnlmzkC+h3yNXMmEVAtqNkFEYS9VQtqMm3r0yrFB9EQPMw7dWScSZ1JBMy7+ryoD4r3ru6wCoTSvbfqdq5yO2MrXqUDU0MkAqqtONkn7m7v1VZiWquFV/vgdphkKwlbCcpMsozMfaSyviIjcUn7qEYWGVkwMir1FdFtgW5HHJoacmVpyHXYlG6d6BbfHAFjQg6pZxIB3Q6jjIx9pLb4JhaPLJXzKVY830hnEo3PQcEsEEad1cipIRIB237YLqzDhnH5oizBNYFY0EiCJpyJIYmAfodBpg19WpUa5rIgci5fdP5XJP/rcF9t8mva1I26WgWTQCyoKue14p/wMD/iEGePHBJIMn5EIUZ5JZbd2mvaULUTYRm+7eloXRMiV/Mjf+KSwvNlcyERsGxwXo0XEgHbDmNbEOYYRpLE0KU1bb+iwogis1LhxPkg0QRnF/z0YEP96Z//vicYUG+MPGhCxNkJYx8Zt2lnPXaQiOpoTIs8+PS0cpKDRECnYBSIaf345hVlzHUlgweJgGUf6WVa36etOwx5wdArMvXNJDB1mHeYZUFwnhDUtLkKhIZG/JmwyILgAt03dyjTwgVCVAtiAzpIBAw7bAJbh0lBWVAvS0c2M9XH8ecgEbApWAQWlKV2L5U4fB8kAqo6yksd1WvWUZhNGGRk6CObgl4gNJTVN+eRPY8jy8oSOMc8SASM+8gk38Q2Qs4LfnmwOQ+77RVMn1attsrIrvgR1iZsArviR1g7fQV8lQqbRBOqgkVgt4RRpk04mwWvo1kwspkTCorTIYo/huJHTCB4eujqo62MaiOOhGWOzHmNxD6jVdNWv2DtI4eBcYNi+ieJgGmH87BHVRU5mjAM0trAYUerDKzKkSGQRuf2ettLvY04FEc0mXBGsMvoXjhVqWeBucOywyDThj5t3WGRXgty2shxiHstU/EkmjhSnbUV8RWU1ZFDDY/khOUg0fRLboNZWjiwvDiqqtmlCdKl6efJ0YRhFIhMJnJMYDiDFIkmjoRlwSwQlhDVN6u0jWDxceQkC8pI2GxMatomC8I9FlNRsArEPtmzh/rc3AqJgMNX3gHfVyV3kAioyzwpnLqZcIUzi+8oxTcgN2a4hK5SQvetFHXzutXM69lwLMqos7hDN7CRWdwjg8LfvOguJAKqZgkXMTcSAUergPuZVvqZcKI4YsJKzIokZhjZVHpVJL1C6hqbqjujFMK4cJJV6qtSWqIUSU4loFYyf8SD5FT6wVnbhUSTfFSXq9xIcPvkdSdNvAz7TF75ZxTn7W1Cr66r2sTLmkkj839EhvF4lpqeRJNwZL98fVh09idIBG2KRqERFIe2U3aXG4mgXtESFkWVnHCmO2VvupEImjWtQqGKqlfVZCyul4S/nSbZL2Jgarxmbi1at1qLCPSpBUU5k7+RCJoV5eT4RiJoUzTIzEiasuVVsVexb9xG8M3I2Ihyr2dqg0TQMfO0cSvFB9ac3dgR2dTnh96Wobvtidoy2VWhT3ILkQg6Zl4JiaQVvjd9x5rZZadtXEgEHWPXdSLmigCdA69q3goSvl2nQVHfFsWsOURFU5F7Ad9F6CfKMdFLTOxrxspIk69/3t1e/5R+7ZBNjto6ZkuaRFBt0UFo6DRpe49i7xFUW2URq0QFmJN8lwxVZm6YOWm7s9LwtjjDpO2OyzmySugjFW13UewOq8q8I4T6Ht+L9FkRyXOOyipTWlaZsKritc1Gsdne2g/KZmffjkRQXhUHUCstP6RGeXo/d9H8emgiu6tJ27s8l/QzqlXRWW2TCMpWyQ9R63mjP0awd88SdRWbOJvMvj0vNra5C4mmsPfOEm6mwiPSFeuVjyWJ6pi52KRoleIGEbLYqjwwS6qMq784u9u6jVKHxP6Awj7GuSl3vQ4SQbMaW6QExAVf3Dif+2fs93mNvTyPsR88n/ujFxJBnaJcXF7GS0PxXsWFqebL0IaPalVNEkKk6MVn5flTV5ehK180DUIDaFVxga/Pg0RTgtLk7L4eJIJ6TWWs62ODokEiDsYVjkcr4sxm8pDBs+bzEh0yePsQsdwWsaDPDzHJyl3Y9xWj8n7ud91IBE2KZrnPMjQWi/ZvyfS6VSe/PJieDiSD7LtKTXuwNMlxwiVbRefdTyLo2NHqI0mHGJ5Ust+91K++D2yj5CwWjzou11XH4c4pRdt0qsumUVqX+sGmvdg0vlo/2HQQm8aK6webluoIqX2p2mo5eh8kghZ9X4lt+f5g6bRtWbEteEvTdhnEamFV5UO2EuRewC1YPmQrM78lEVTfGvN1iURQncskuVOw32qdvjWK3BoFlC1yvrIVeWXrNOtMR24NaLFyNFyvK05eV5ypHA1nOt+kqY5SqXI0XE3PKE1PrNkpq8rrXsidsX/3uuXnh54xklW9UMZYP0RKfuO8kAha1dnzI8pxGc/KHEdPb6DybniMd8PqdS46n8VIBC1KF1Y8DBquo6RcL+RyfLgea9LP4F6ewTFvVm/PRd66UQrU0QOfsMnjM7KUyiXlfM/N8p6LoxvNogmTPPYi6lf9O4Akr89dwVk9amcZiShSi7wh42gQwPrREAyA6t2aO4M3EgHVtFWew7spVfVWXuVRGzdJraoADvLAHDpUT9NTfaMVUqt+Jo7yTIwF1bpDL4+9vk+r3pA5ISYII+OfGbAlcNP6RiJgUU8ITp4QnGlWVapN3lFwns2qNw0Ol2cSAdXPS6xMa/u0VUGpq22fVr2GWHmZsMU0p15DsoyEcTWnyvUqP3eB0pvbOw/zoe5CImDa2xJeHgSQjTSn+tJOSiA4b3Oq9Rwld0OUbNx/54ZjqKvhGDDSq1bl7AeRCBhV21BSs74V/qnBvPHnzyaGbBr/KoD7HV5+3NVPFKH9878ehAahAbQOyp0dK79CwpE1bGmnTvo+CJEtBk2z0AwaFfVSSHQVx7Gq9ZonT30IYw1erqg8ysG6WwqKZnnPQ6LQ0vjusm8xJ5o5K1rlSoC3tTRWxUF/Nqu6BNZ2FlYxgGu35aGnaWsTkgg6djNfgvjqpPcczAs7Jjo7u9IEQ/BtuSo6k3ISTStW0ygU54P/g1Y1W0BZWkA4nzK+O725ys/zEFqc5dvx/zRSuma1KAAA", "PAD_UFES_20": "..."}\nIMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\ndef sha256_file(path: Path, block: int = 1 << 20) -> str:\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(block), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\ndef decode_manifests() -> Dict[str, pd.DataFrame]:\n    frames = {}\n    for name, payload in MANIFEST_PAYLOADS.items():\n        raw = gzip.decompress(base64.b64decode(payload.encode(\'ascii\')))\n        df = pd.read_csv(io.BytesIO(raw))\n        df[\'image_id_norm\'] = df[\'image_id_norm\'].astype(str).str.strip().str.lower()\n        df[\'mapped_class\'] = df[\'mapped_class\'].astype(str).str.strip()\n        df[\'group_id\'] = df[\'group_id\'].astype(str).str.strip()\n        df[\'manifest_sha256\'] = sha256_bytes(raw)\n        frames[name] = df\n    return frames\n\ndef build_model(family: str) -> nn.Module:\n    if family == "B0":\n        m = models.efficientnet_b0(weights=None)\n    elif family == "B3":\n        m = models.efficientnet_b3(weights=None)\n    else:\n        raise ValueError(f"Unknown model family: {family}")\n    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)\n    return m\n\ndef state_dict_from_checkpoint(obj):\n    if isinstance(obj, dict):\n        for key in (\'model_state_dict\', \'state_dict\', \'model\', \'ema_state_dict\', \'model_ema_state_dict\'):\n            value = obj.get(key)\n            if isinstance(value, dict) and value and all(torch.is_tensor(v) for v in value.values()):\n                return value\n        if obj and all(torch.is_tensor(v) for v in obj.values()):\n            return obj\n    raise ValueError(\'Could not locate a tensor state_dict inside checkpoint.\')\n\ndef normalize_state_dict_keys(state):\n    keys = list(state.keys())\n    if keys and all(str(k).startswith(\'module.\') for k in keys):\n        return {str(k)[7:]: v for k, v in state.items()}\n    if keys and all(str(k).startswith(\'_orig_mod.\') for k in keys):\n        return {str(k)[10:]: v for k, v in state.items()}\n    return state\n\ndef find_exact_checkpoints() -> Dict[str, Path]:\n    target_to_name = {v: k for k, v in CHECKPOINT_HASHES.items()}\n    found: Dict[str, Path] = {}\n    extracted = OUTPUT_ROOT / \'verified_checkpoints\'\n    extracted.mkdir(parents=True, exist_ok=True)\n\n    for p in INPUT_ROOT.rglob(\'*\'):\n        if not p.is_file():\n            continue\n        if p.suffix.lower() not in {\'.pth\', \'.pt\', \'.ckpt\'}:\n            continue\n        try:\n            digest = sha256_file(p)\n        except Exception:\n            continue\n        if digest in target_to_name:\n            name = target_to_name[digest]\n            found[name] = p\n            print(f\'[CHECKPOINT] Found exact {name}: {p}\')\n\n    missing = set(CHECKPOINT_HASHES) - set(found)\n    if missing:\n        for zp in INPUT_ROOT.rglob(\'*.zip\'):\n            try:\n                with zipfile.ZipFile(zp) as z:\n                    for member in z.infolist():\n                        if not member.filename.lower().endswith((\'.pth\', \'.pt\', \'.ckpt\')):\n                            continue\n                        data = z.read(member)\n                        digest = sha256_bytes(data)\n                        name = target_to_name.get(digest)\n                        if name is None or name in found:\n                            continue\n                        outp = extracted / f\'{name}_{digest[:12]}.pth\'\n                        outp.write_bytes(data)\n                        found[name] = outp\n                        print(f\'[CHECKPOINT] Extracted exact {name} from {zp.name}:{member.filename}\')\n            except zipfile.BadZipFile:\n                continue\n            if set(found) == set(CHECKPOINT_HASHES):\n                break\n\n    if set(found) != set(CHECKPOINT_HASHES):\n        raise RuntimeError(\n            \'Could not find all six exact frozen PAPER-EXP-01A/01B checkpoints. \'\n            f\'Found={sorted(found)}. Expected hashes={CHECKPOINT_HASHES}. \'\n            \'Attach PAPEREXP01A B0 Recovery and PAPEREXP01B B3 Recovery datasets/ZIPs.\'\n        )\n\n    verified = {}\n    for name, path in found.items():\n        digest = sha256_file(path)\n        if digest != CHECKPOINT_HASHES[name]:\n            raise RuntimeError(f\'Hash mismatch for {name}: {digest}\')\n        verified[name] = path\n    return verified\n\ndef load_model(checkpoint_path: Path, device: torch.device, family: str) -> nn.Module:\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    if isinstance(checkpoint, dict) and "class_order" in checkpoint:\n        class_order = tuple(checkpoint["class_order"])\n        if class_order != CLASS_NAMES:\n            raise RuntimeError(f"Class order mismatch: {class_order} != {CLASS_NAMES}")\n    state = normalize_state_dict_keys(state_dict_from_checkpoint(checkpoint))\n    model = build_model(family)\n    model.load_state_dict(state, strict=True)\n    model = model.to(device).eval()\n    for name, tensor in model.state_dict().items():\n        if torch.is_floating_point(tensor) and not torch.isfinite(tensor).all():\n            raise RuntimeError(f"Non-finite model tensor: {name}")\n    return model\n\ndef make_validation_transform():\n    return transforms.Compose([\n        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),\n        transforms.ToTensor(),\n        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n    ])\n\ndef path_matches_dataset(path: Path, tokens: Tuple[str, ...]) -> bool:\n    text = str(path).lower()\n    return any(token in text for token in tokens)\n\ndef build_dataset_image_index(dataset_name: str):\n    tokens = DATASET_CONFIG[dataset_name][\'tokens\']\n    by_stem = defaultdict(list)\n    by_name = defaultdict(list)\n    total = 0\n    for p in INPUT_ROOT.rglob(\'*\'):\n        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:\n            continue\n        if not path_matches_dataset(p, tokens):\n            continue\n        by_stem[p.stem.lower()].append(p)\n        by_name[p.name.lower()].append(p)\n        total += 1\n    if total == 0:\n        raise RuntimeError(f\'No images found for {dataset_name}. Expected a Kaggle input path containing one of {tokens}.\')\n    print(f\'[DATASET] {dataset_name}: indexed {total} image files\')\n    return by_stem, by_name\n\ndef resolve_manifest_paths(dataset_name: str, manifest: pd.DataFrame) -> pd.DataFrame:\n    by_stem, by_name = build_dataset_image_index(dataset_name)\n    out = manifest.copy()\n    resolved = []\n    for _, row in out.iterrows():\n        hint = str(row[\'image_hint\']).strip()\n        image_id = str(row[\'image_id_norm\']).strip().lower()\n        candidates = []\n        basename = Path(hint).name.lower()\n        if basename:\n            candidates.extend(by_name.get(basename, []))\n        candidates.extend(by_stem.get(image_id, []))\n        seen = set()\n        unique = []\n        for p in candidates:\n            sp = str(p)\n            if sp not in seen:\n                seen.add(sp)\n                unique.append(p)\n        if dataset_name == \'Derm7pt\' and len(unique) > 1:\n            normalized_hint = hint.replace(\'\\\\\', \'/\').lower()\n            suffix_matches = [p for p in unique if str(p).replace(\'\\\\\', \'/\').lower().endswith(normalized_hint)]\n            if len(suffix_matches) == 1:\n                unique = suffix_matches\n        if len(unique) != 1:\n            raise RuntimeError(\n                f"{dataset_name}: could not uniquely resolve image id={image_id}, hint={hint!r}. candidates={unique[:8]}"\n            )\n        resolved.append(str(unique[0]))\n    out[\'image_path\'] = resolved\n    if out[\'image_path\'].duplicated().any():\n        dup = out.loc[out[\'image_path\'].duplicated(False), [\'image_id_norm\', \'image_path\']]\n        raise RuntimeError(f\'{dataset_name}: duplicate resolved paths in frozen cohort:\\n{dup.head(20)}\')\n    if len(out) != DATASET_CONFIG[dataset_name][\'expected_n\']:\n        raise RuntimeError(f"{dataset_name}: cohort size changed: {len(out)} != {DATASET_CONFIG[dataset_name][\'expected_n\']}")\n    expected_classes = set(DATASET_CONFIG[dataset_name][\'supported_classes\'])\n    actual_classes = set(out[\'mapped_class\'])\n    if not actual_classes.issubset(expected_classes):\n        raise RuntimeError(f"{dataset_name}: unsupported true labels in manifest: {sorted(actual_classes - expected_classes)}")\n    out[\'label\'] = out[\'mapped_class\'].map(CLASS_TO_INDEX).astype(int)\n    return out.reset_index(drop=True)\n\nclass ExternalDataset(Dataset):\n    def __init__(self, frame: pd.DataFrame):\n        self.frame = frame.reset_index(drop=True)\n        self.transform = make_validation_transform()\n    def __len__(self):\n        return len(self.frame)\n    def __getitem__(self, idx):\n        row = self.frame.iloc[idx]\n        path = Path(row[\'image_path\'])\n        try:\n            with Image.open(path) as im:\n                im = ImageOps.exif_transpose(im).convert(\'RGB\')\n                x = self.transform(im)\n        except Exception as e:\n            raise RuntimeError(f\'Could not read {path}\') from e\n        return x, int(idx)\n\ndef make_loader(frame: pd.DataFrame, batch_size: int, workers: int):\n    kwargs = dict(\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=max(0, int(workers)),\n        pin_memory=torch.cuda.is_available(),\n        drop_last=False,\n    )\n    if workers > 0:\n        kwargs[\'persistent_workers\'] = True\n        kwargs[\'prefetch_factor\'] = 2\n    return DataLoader(ExternalDataset(frame), **kwargs)\n\n@contextlib.contextmanager\ndef autocast_context(device):\n    if device.type == \'cuda\':\n        with torch.autocast(device_type=\'cuda\', dtype=torch.float16):\n            yield\n    else:\n        yield\n\n@torch.no_grad()\ndef tta7_predict(model, images):\n    views = (\n        images,\n        torch.flip(images, dims=(3,)),\n        torch.flip(images, dims=(2,)),\n        torch.flip(images, dims=(2, 3)),\n        torch.rot90(images, 1, dims=(2, 3)),\n        torch.rot90(images, 2, dims=(2, 3)),\n        torch.rot90(images, 3, dims=(2, 3)),\n    )\n    total = None\n    for view in views:\n        logits = model(view)\n        p = F.softmax(logits.float(), dim=1)\n        total = p if total is None else total + p\n    return total / 7.0\n\ndef _append_numerical_fallback_event(event):\n    log_path = OUTPUT_ROOT / "numerical_precision_fallback_log.json"\n    events = []\n    if log_path.exists():\n        try:\n            events = json.loads(log_path.read_text(encoding="utf-8"))\n        except Exception:\n            events = []\n    events.append(event)\n    log_path.write_text(json.dumps(events, indent=2), encoding="utf-8")\n\n\ndef predict_dataset(model, frame: pd.DataFrame, batch_size: int, workers: int, device: torch.device, desc: str) -> np.ndarray:\n    """\n    Fixed TTA7 inference with a numerical-safety fallback.\n\n    Primary path is exactly the predeclared CUDA FP16 autocast inference.\n    If (and only if) a row becomes non-finite, that row is recomputed using\n    the identical frozen model and identical seven TTA views in FP32.\n\n    This fallback is label-blind and changes no model, checkpoint, view,\n    ensemble weight, threshold or calibration setting.\n    """\n    loader = make_loader(frame, batch_size, workers)\n    probs = []\n    indices = []\n\n    with torch.inference_mode():\n        for images, batch_idx in tqdm(loader, desc=desc):\n            images = images.to(device, non_blocking=True)\n\n            if not torch.isfinite(images).all():\n                bad_input = (~torch.isfinite(images).flatten(1).all(dim=1)).cpu()\n                bad_ids = batch_idx[bad_input].tolist()\n                raise RuntimeError(\n                    f"Non-finite normalized INPUT tensor in {desc}; row indices={bad_ids}"\n                )\n\n            # Primary, protocol-consistent inference path.\n            with autocast_context(device):\n                p = tta7_predict(model, images)\n\n            bad_mask = ~torch.isfinite(p).all(dim=1)\n\n            # Minimal numerical repair: only unstable rows are recomputed in FP32.\n            if bool(bad_mask.any()):\n                bad_cpu = bad_mask.detach().cpu()\n                bad_indices = batch_idx[bad_cpu].tolist()\n                bad_count = int(bad_mask.sum().item())\n\n                print(\n                    f"[NUMERICAL FALLBACK] {desc}: {bad_count} row(s) "\n                    f"non-finite under FP16; rerunning only those rows in FP32. "\n                    f"indices={bad_indices}"\n                )\n\n                bad_images = images[bad_mask].float()\n                with torch.autocast(device_type="cuda", enabled=False):\n                    p32 = tta7_predict(model, bad_images)\n\n                if not torch.isfinite(p32).all():\n                    raise RuntimeError(\n                        f"FP32 fallback also produced non-finite probabilities in {desc}; "\n                        f"row indices={bad_indices}"\n                    )\n\n                if not torch.allclose(\n                    p32.sum(dim=1),\n                    torch.ones(p32.size(0), device=p32.device),\n                    atol=2e-5,\n                    rtol=0,\n                ):\n                    raise RuntimeError(\n                        f"FP32 fallback probability rows do not sum to 1 in {desc}; "\n                        f"row indices={bad_indices}"\n                    )\n\n                p = p.clone()\n                p[bad_mask] = p32\n\n                _append_numerical_fallback_event({\n                    "description": desc,\n                    "row_indices": [int(x) for x in bad_indices],\n                    "n_rows": bad_count,\n                    "primary_precision": "CUDA autocast float16",\n                    "fallback_precision": "float32",\n                    "model_changed": False,\n                    "tta_changed": False,\n                    "weights_changed": False,\n                    "threshold_changed": False,\n                    "calibration_changed": False,\n                    "labels_used_for_fallback": False,\n                })\n\n            if not torch.isfinite(p).all():\n                raise RuntimeError(f"Non-finite probabilities remain after fallback in {desc}")\n\n            if not torch.allclose(\n                p.sum(dim=1),\n                torch.ones(p.size(0), device=p.device),\n                atol=2e-5,\n                rtol=0,\n            ):\n                raise RuntimeError(f"Probability rows do not sum to 1 in {desc}")\n\n            probs.append(p.cpu().numpy())\n            indices.append(batch_idx.numpy())\n\n    probs = np.concatenate(probs, axis=0)\n    indices = np.concatenate(indices, axis=0)\n    order = np.argsort(indices)\n    if not np.array_equal(indices[order], np.arange(len(frame))):\n        raise RuntimeError(f"Incomplete/duplicated prediction indices in {desc}")\n    return probs[order]\n\ndef save_model_prediction(dataset_name: str, frame: pd.DataFrame, model_name: str, probs: np.ndarray):\n    outdir = PRED_ROOT / model_name\n    outdir.mkdir(parents=True, exist_ok=True)\n    pred = frame[[\'image_id_norm\', \'mapped_class\', \'group_id\', \'image_path\', \'label\']].copy()\n    pred[\'model_name\'] = model_name\n    pred[\'predicted_label\'] = probs.argmax(axis=1)\n    pred[\'predicted_class\'] = [CLASS_NAMES[i] for i in pred[\'predicted_label\']]\n    for i, c in enumerate(CLASS_NAMES):\n        pred[f\'prob_{c}\'] = probs[:, i]\n    outp = outdir / f\'{dataset_name}_tta7.csv\'\n    pred.to_csv(outp, index=False)\n    parr = pred[[f\'prob_{c}\' for c in CLASS_NAMES]].to_numpy(float)\n    if not np.isfinite(parr).all():\n        raise RuntimeError(f\'Non-finite saved probabilities: {outp}\')\n    if not np.allclose(parr.sum(axis=1), 1.0, atol=2e-5, rtol=0):\n        raise RuntimeError(f\'Probability rows do not sum to 1: {outp}\')\n    print(f\'[SAVED] {outp}\')\n    return outp\n\ndef metric_bundle(y: np.ndarray, probs: np.ndarray, supported_classes: Tuple[str, ...]):\n    pred = probs.argmax(axis=1)\n    supported_indices = [CLASS_TO_INDEX[c] for c in supported_classes]\n    accuracy = float(accuracy_score(y, pred))\n    macro_f1 = float(f1_score(y, pred, labels=supported_indices, average=\'macro\', zero_division=0))\n    precision_arr, recall_arr, f1_arr, support_arr = precision_recall_fscore_support(\n        y, pred, labels=list(range(NUM_CLASSES)), zero_division=0\n    )\n    recalls = {c: float(recall_arr[i]) for i, c in enumerate(CLASS_NAMES)}\n    precisions = {c: float(precision_arr[i]) for i, c in enumerate(CLASS_NAMES)}\n    f1s = {c: float(f1_arr[i]) for i, c in enumerate(CLASS_NAMES)}\n    supports = {c: int(support_arr[i]) for i, c in enumerate(CLASS_NAMES)}\n    balanced_accuracy = float(np.mean([recalls[c] for c in supported_classes]))\n    melanoma_recall = recalls[\'MEL\']\n    aucs = {}\n    for c in supported_classes:\n        idx = CLASS_TO_INDEX[c]\n        binary = (y == idx).astype(int)\n        if binary.min() == binary.max():\n            aucs[c] = None\n        else:\n            aucs[c] = float(roc_auc_score(binary, probs[:, idx]))\n    finite_aucs = [v for v in aucs.values() if v is not None and math.isfinite(v)]\n    macro_auc = float(np.mean(finite_aucs)) if finite_aucs else float(\'nan\')\n    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))\n    return {\n        \'n\': int(len(y)),\n        \'accuracy\': accuracy,\n        \'macro_f1_supported_classes\': macro_f1,\n        \'balanced_accuracy_supported_classes\': balanced_accuracy,\n        \'melanoma_recall\': melanoma_recall,\n        \'macro_auc_supported_classes_ovr\': macro_auc,\n        \'per_class_auc_supported_only\': aucs,\n        \'per_class_recall\': recalls,\n        \'per_class_precision\': precisions,\n        \'per_class_f1\': f1s,\n        \'per_class_support\': supports,\n        \'confusion_matrix_7x7\': cm.tolist(),\n        \'predicted_DF_count\': int((pred == CLASS_TO_INDEX[\'DF\']).sum()),\n        \'predicted_DF_rate\': float((pred == CLASS_TO_INDEX[\'DF\']).mean()),\n        \'predicted_VASC_count\': int((pred == CLASS_TO_INDEX[\'VASC\']).sum()),\n        \'predicted_VASC_rate\': float((pred == CLASS_TO_INDEX[\'VASC\']).mean()),\n    }\n\ndef bootstrap_ci(frame: pd.DataFrame, probs: np.ndarray, supported_classes: Tuple[str, ...], iterations: int, seed: int):\n    y = frame[\'label\'].to_numpy(int)\n    groups = frame[\'group_id\'].astype(str).to_numpy()\n    unique_groups = np.unique(groups)\n    positions = {g: np.flatnonzero(groups == g) for g in unique_groups}\n    metric_names = (\n        \'accuracy\',\n        \'macro_f1_supported_classes\',\n        \'balanced_accuracy_supported_classes\',\n        \'melanoma_recall\',\n        \'macro_auc_supported_classes_ovr\',\n    )\n    values = {m: [] for m in metric_names}\n    rng = np.random.default_rng(seed)\n    supported_idx = {CLASS_TO_INDEX[c] for c in supported_classes}\n    for _ in tqdm(range(iterations), desc=\'bootstrap\', leave=False):\n        sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)\n        idx = np.concatenate([positions[g] for g in sampled])\n        if not supported_idx.issubset(set(y[idx].tolist())):\n            continue\n        m = metric_bundle(y[idx], probs[idx], supported_classes)\n        for name in metric_names:\n            val = float(m[name])\n            if math.isfinite(val):\n                values[name].append(val)\n    result = {\'iterations_requested\': int(iterations), \'group_count\': int(len(unique_groups)), \'metrics\': {}}\n    for name, vals in values.items():\n        arr = np.asarray(vals, dtype=float)\n        result[\'metrics\'][name] = {\n            \'valid_iterations\': int(len(arr)),\n            \'mean\': float(arr.mean()) if len(arr) else None,\n            \'ci95_low\': float(np.quantile(arr, 0.025)) if len(arr) else None,\n            \'ci95_high\': float(np.quantile(arr, 0.975)) if len(arr) else None,\n        }\n    return result\n\n\n\nEXPECTED_EXTERNAL_MANIFEST_SHA256 = {\'DERM12345\': \'9becd1ea91602dca071bb7f561011d0fc6f46266b86d148ac3b3d463db909453\', \'BCN20000\': \'d9eeacfe22638dca39eb052ddad78c418f783f62f258551cb5767c734fa0debd\', \'Derm7pt\': \'47e092f799d8c4c97cd94ccb3116995ad03e00b8e249f24b1edfdbcaf9dc5299\', \'PAD_UFES_20\': \'18f07c4079735c533c1ab8ba17e77377f43143f052fcb4bde468cc255b453bef\'}\nSOURCE_RECOVERY_SHA256 = {\'PAPEREXP01A_B0_Recovery_SHA256\': \'778060755a8be16d8ab7bb21055d50a6f0a641a763f19caf30da399096192668\', \'PAPEREXP01B_B3_Recovery_SHA256\': \'2a25768706d4bd7ee7ce6f6982ceedd6b0b97de936cf3739f68cdba7e2bc9ae8\'}\n\n\ndef _family_for_model(model_name: str) -> str:\n    if model_name in B0_MODEL_NAMES:\n        return "B0"\n    if model_name in B3_MODEL_NAMES:\n        return "B3"\n    raise KeyError(model_name)\n\n\ndef _manifest_hash_guard(manifests):\n    for name, df in manifests.items():\n        actual = str(df["manifest_sha256"].iloc[0])\n        expected = EXPECTED_EXTERNAL_MANIFEST_SHA256[name]\n        if actual != expected:\n            raise RuntimeError(\n                f"External manifest hash changed for {name}: expected={expected}, actual={actual}"\n            )\n\n\ndef _existing_prediction_is_valid(path: Path, frame: pd.DataFrame, model_name: str) -> bool:\n    if not path.exists():\n        return False\n    try:\n        df = pd.read_csv(path)\n        required = {\n            "image_id_norm", "mapped_class", "group_id", "image_path", "label",\n            "model_name", "predicted_label", "predicted_class",\n        } | {f"prob_{c}" for c in CLASS_NAMES}\n        if not required.issubset(df.columns):\n            return False\n        if len(df) != len(frame):\n            return False\n        key_cols = ["image_id_norm", "mapped_class", "group_id", "image_path", "label"]\n        if not df[key_cols].equals(frame[key_cols].reset_index(drop=True)):\n            return False\n        if not df["model_name"].astype(str).eq(model_name).all():\n            return False\n        parr = df[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy(float)\n        if not np.isfinite(parr).all():\n            return False\n        if not np.allclose(parr.sum(axis=1), 1.0, atol=2e-5, rtol=0):\n            return False\n        return True\n    except Exception:\n        return False\n\n\ndef run_models(workers: int):\n    manifests = decode_manifests()\n    _manifest_hash_guard(manifests)\n    frames = {name: resolve_manifest_paths(name, df) for name, df in manifests.items()}\n    checkpoints = find_exact_checkpoints()\n\n    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n    if device.type != "cuda":\n        raise RuntimeError("Enable a Kaggle GPU accelerator.")\n    print("Using GPU:", torch.cuda.get_device_name(0))\n    print(\n        "[NUMERICAL POLICY] Primary inference = CUDA FP16 autocast. "\n        "Only non-finite rows are rerun with identical fixed TTA7 in FP32."\n    )\n    print(\n        "[RESUME POLICY] Existing prediction CSVs are reused ONLY after exact "\n        "row/key/model/probability validation."\n    )\n\n    model_summary = {\n        "code_version": CODE_VERSION,\n        "device": str(device),\n        "locked_system": (\n            "B0 family: 3 seeds equal 1/3 TTA7; "\n            "B3 family: 3 seeds equal 1/3 TTA7; "\n            "final fixed fusion: 0.50 B0 + 0.50 B3"\n        ),\n        "numerical_precision_policy": (\n            "FP16 autocast primary; row-level FP32 fallback only when FP16 "\n            "probabilities are non-finite; no label access"\n        ),\n        "resume_policy": (\n            "reuse only existing CSVs that pass exact schema/key/model/finite/"\n            "probability-sum validation"\n        ),\n        "models": {},\n    }\n\n    for model_name in ALL_MODEL_NAMES:\n        family = _family_for_model(model_name)\n\n        missing_datasets = []\n        for dataset_name, frame in frames.items():\n            existing = PRED_ROOT / model_name / f"{dataset_name}_tta7.csv"\n            if _existing_prediction_is_valid(existing, frame, model_name):\n                print(f"[RESUME PASS] {model_name} | {dataset_name} already valid; skipping.")\n            else:\n                missing_datasets.append(dataset_name)\n\n        model_summary["models"][model_name] = {\n            "family": family,\n            "checkpoint_path": str(checkpoints[model_name]),\n            "checkpoint_sha256": sha256_file(checkpoints[model_name]),\n            "datasets": {},\n        }\n\n        # Record already-complete datasets.\n        for dataset_name, frame in frames.items():\n            existing = PRED_ROOT / model_name / f"{dataset_name}_tta7.csv"\n            if dataset_name not in missing_datasets:\n                df = pd.read_csv(existing)\n                pcols = [f"prob_{c}" for c in CLASS_NAMES]\n                parr = df[pcols].to_numpy(float)\n                model_summary["models"][model_name]["datasets"][dataset_name] = {\n                    "n": int(len(frame)),\n                    "probability_shape": list(parr.shape),\n                    "finite": True,\n                    "probability_sum_max_abs_error": float(\n                        np.max(np.abs(parr.sum(axis=1) - 1.0))\n                    ),\n                    "reused_from_valid_existing_output": True,\n                }\n\n        if not missing_datasets:\n            print(f"[MODEL COMPLETE] {model_name}: all external datasets already valid.")\n            continue\n\n        print("\\n" + "=" * 88)\n        print("MODEL", model_name, "| FAMILY", family, "| missing:", missing_datasets)\n        print("=" * 88)\n\n        model = load_model(checkpoints[model_name], device, family)\n\n        for dataset_name in missing_datasets:\n            frame = frames[dataset_name]\n            probs = predict_dataset(\n                model,\n                frame,\n                MODEL_CONFIG[model_name]["batch_size"],\n                workers,\n                device,\n                f"{model_name} | {dataset_name} | fixed TTA7",\n            )\n            save_model_prediction(dataset_name, frame, model_name, probs)\n            model_summary["models"][model_name]["datasets"][dataset_name] = {\n                "n": int(len(frame)),\n                "probability_shape": list(probs.shape),\n                "finite": bool(np.isfinite(probs).all()),\n                "probability_sum_max_abs_error": float(\n                    np.max(np.abs(probs.sum(axis=1) - 1.0))\n                ),\n                "reused_from_valid_existing_output": False,\n            }\n            make_recovery_zip()\n\n        del model\n        torch.cuda.empty_cache()\n\n    (OUTPUT_ROOT / "model_prediction_summary.json").write_text(\n        json.dumps(model_summary, indent=2), encoding="utf-8"\n    )\n\ndef _per_class_extended(y, probs, supported_classes):\n    pred = probs.argmax(axis=1)\n    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y, pred, labels=list(range(NUM_CLASSES)), zero_division=0\n    )\n    rows = []\n    for i, c in enumerate(CLASS_NAMES):\n        tp = int(cm[i, i])\n        fn = int(cm[i, :].sum() - tp)\n        fp = int(cm[:, i].sum() - tp)\n        tn = int(cm.sum() - tp - fn - fp)\n        binary = (y == i).astype(int)\n        if binary.min() == binary.max():\n            auc = None\n            ap = None\n        else:\n            auc = float(roc_auc_score(binary, probs[:, i]))\n            ap = float(average_precision_score(binary, probs[:, i]))\n        rows.append({\n            "class_name": c,\n            "true_class_supported_in_domain": bool(c in supported_classes),\n            "precision": float(precision[i]),\n            "recall": float(recall[i]),\n            "specificity": float(tn / (tn + fp)) if (tn + fp) else None,\n            "f1": float(f1[i]),\n            "support": int(support[i]),\n            "roc_auc": auc,\n            "average_precision": ap,\n        })\n    return pd.DataFrame(rows), cm\n\n\ndef _save_figure(fig, fig_dir: Path, stem: str):\n    fig.tight_layout()\n    fig.savefig(fig_dir / f"{stem}.png", dpi=300, bbox_inches="tight")\n    fig.savefig(fig_dir / f"{stem}.pdf", bbox_inches="tight")\n    fig.savefig(fig_dir / f"{stem}.svg", bbox_inches="tight")\n    plt.close(fig)\n\n\ndef _publication_package(frames, final_probs_by_domain, summary_df):\n    fig_dir = OUTPUT_ROOT / "figures"\n    tab_dir = OUTPUT_ROOT / "tables"\n    src_dir = OUTPUT_ROOT / "figure_source_data"\n    for p in (fig_dir, tab_dir, src_dir):\n        p.mkdir(parents=True, exist_ok=True)\n\n    # Main domain-level table.\n    primary_cols = [\n        "dataset", "n", "accuracy", "accuracy_ci95_low", "accuracy_ci95_high",\n        "macro_f1", "macro_f1_ci95_low", "macro_f1_ci95_high",\n        "balanced_accuracy", "balanced_accuracy_ci95_low", "balanced_accuracy_ci95_high",\n        "melanoma_recall", "melanoma_recall_ci95_low", "melanoma_recall_ci95_high",\n        "macro_auc", "macro_auc_ci95_low", "macro_auc_ci95_high",\n    ]\n    summary_df[primary_cols].to_csv(\n        tab_dir / "Table1_external_primary_metrics_with_95CI.csv", index=False\n    )\n\n    # Figure: external primary metrics by domain.\n    metrics = [\n        ("accuracy", "Accuracy"),\n        ("macro_f1", "Macro-F1"),\n        ("balanced_accuracy", "Balanced accuracy"),\n        ("melanoma_recall", "MEL recall"),\n        ("macro_auc", "Macro AUC"),\n    ]\n    x = np.arange(len(summary_df))\n    width = 0.15\n    fig, ax = plt.subplots(figsize=(12, 6.5))\n    offsets = np.linspace(-2, 2, len(metrics)) * width\n    for off, (col, label) in zip(offsets, metrics):\n        ax.bar(x + off, summary_df[col], width, label=label)\n    ax.set_xticks(x, summary_df["dataset"])\n    ax.set_ylim(0, 1.0)\n    ax.set_ylabel("Score")\n    ax.set_title("Frozen PAPER-EXP-01C zero-shot external benchmark performance")\n    ax.legend(fontsize=8)\n    _save_figure(fig, fig_dir, "Fig1_external_domain_primary_metrics")\n    summary_df.to_csv(src_dir / "Fig1_external_domain_primary_metrics_source.csv", index=False)\n\n    # Figure: internal reference vs equal-domain external mean.\n    equal_mean = {\n        "accuracy": float(summary_df["accuracy"].mean()),\n        "macro_f1": float(summary_df["macro_f1"].mean()),\n        "balanced_accuracy": float(summary_df["balanced_accuracy"].mean()),\n        "melanoma_recall": float(summary_df["melanoma_recall"].mean()),\n        "macro_auc": float(summary_df["macro_auc"].mean()),\n    }\n    compare = pd.DataFrame({\n        "metric": [m[0] for m in metrics],\n        "PAPEREXP01C_internal": [CLEAN_INTERNAL_REFERENCE[m[0]] for m in metrics],\n        "equal_domain_external_mean": [equal_mean[m[0]] for m in metrics],\n    })\n    compare["external_minus_internal"] = (\n        compare["equal_domain_external_mean"] - compare["PAPEREXP01C_internal"]\n    )\n    compare.to_csv(tab_dir / "Table2_internal_vs_external_equal_domain_mean.csv", index=False)\n    xv = np.arange(len(compare))\n    w = 0.36\n    fig, ax = plt.subplots(figsize=(11, 6))\n    ax.bar(xv - w/2, compare["PAPEREXP01C_internal"], w, label="PAPER-EXP-01C clean internal")\n    ax.bar(xv + w/2, compare["equal_domain_external_mean"], w, label="Equal-domain external mean")\n    ax.set_xticks(xv, [m.replace("_", "\\n") for m in compare["metric"]])\n    ax.set_ylim(0, 1.0)\n    ax.set_ylabel("Score")\n    ax.set_title("Internal lesion-safe benchmark vs frozen external mean")\n    ax.legend()\n    _save_figure(fig, fig_dir, "Fig2_internal_vs_external_mean")\n\n    for dataset_name, frame in frames.items():\n        probs = final_probs_by_domain[dataset_name]\n        y = frame["label"].to_numpy(int)\n        supported = tuple(DATASET_CONFIG[dataset_name]["supported_classes"])\n        pc, cm = _per_class_extended(y, probs, supported)\n        pc.to_csv(tab_dir / f"{dataset_name}_per_class_metrics.csv", index=False)\n\n        # Class distribution.\n        dist = (\n            frame.groupby(["mapped_class", "label"], as_index=False)\n            .agg(images=("image_id_norm", "size"), groups=("group_id", "nunique"))\n            .sort_values("label")\n        )\n        dist.to_csv(tab_dir / f"{dataset_name}_class_distribution.csv", index=False)\n\n        # Raw + row-normalized confusion matrices.\n        cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)\n        cm_df.to_csv(src_dir / f"{dataset_name}_confusion_matrix_counts.csv")\n        den = cm.sum(axis=1, keepdims=True)\n        cm_norm = np.divide(cm, den, out=np.zeros_like(cm, dtype=float), where=den != 0)\n        pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(\n            src_dir / f"{dataset_name}_confusion_matrix_row_normalized.csv"\n        )\n        for data, suffix, integer in (\n            (cm, "counts", True),\n            (cm_norm, "row_normalized", False),\n        ):\n            fig, ax = plt.subplots(figsize=(8, 7))\n            im = ax.imshow(data, aspect="auto")\n            ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")\n            ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)\n            ax.set_xlabel("Predicted class")\n            ax.set_ylabel("True class")\n            ax.set_title(f"{dataset_name}: confusion matrix ({suffix})")\n            for i in range(NUM_CLASSES):\n                for j in range(NUM_CLASSES):\n                    txt = f"{int(data[i, j])}" if integer else f"{data[i, j]:.2f}"\n                    ax.text(j, i, txt, ha="center", va="center", fontsize=7)\n            fig.colorbar(im, ax=ax)\n            _save_figure(\n                fig, fig_dir,\n                f"{dataset_name}_confusion_matrix_{suffix}"\n            )\n\n        # ROC and PR curves for supported classes only.\n        roc_rows = []\n        pr_rows = []\n        fig_roc, ax_roc = plt.subplots(figsize=(8, 7))\n        fig_pr, ax_pr = plt.subplots(figsize=(8, 7))\n        for c in supported:\n            i = CLASS_TO_INDEX[c]\n            binary = (y == i).astype(int)\n            if binary.min() == binary.max():\n                continue\n            fpr, tpr, _ = roc_curve(binary, probs[:, i])\n            precision, recall, _ = precision_recall_curve(binary, probs[:, i])\n            auc = roc_auc_score(binary, probs[:, i])\n            ap = average_precision_score(binary, probs[:, i])\n            ax_roc.plot(fpr, tpr, label=f"{c} (AUC {auc:.3f})")\n            ax_pr.plot(recall, precision, label=f"{c} (AP {ap:.3f})")\n            for a, b in zip(fpr, tpr):\n                roc_rows.append({\n                    "class_name": c,\n                    "fpr": float(a),\n                    "tpr": float(b),\n                    "roc_auc": float(auc),\n                })\n            for a, b in zip(recall, precision):\n                pr_rows.append({\n                    "class_name": c,\n                    "recall": float(a),\n                    "precision": float(b),\n                    "average_precision": float(ap),\n                })\n\n        ax_roc.plot([0, 1], [0, 1], linestyle="--")\n        ax_roc.set_xlabel("False positive rate")\n        ax_roc.set_ylabel("True positive rate")\n        ax_roc.set_title(f"{dataset_name}: one-vs-rest ROC")\n        ax_roc.legend(fontsize=8)\n        _save_figure(fig_roc, fig_dir, f"{dataset_name}_ROC")\n\n        ax_pr.set_xlabel("Recall")\n        ax_pr.set_ylabel("Precision")\n        ax_pr.set_title(f"{dataset_name}: precision-recall")\n        ax_pr.legend(fontsize=8)\n        _save_figure(fig_pr, fig_dir, f"{dataset_name}_PR")\n\n        pd.DataFrame(roc_rows).to_csv(\n            src_dir / f"{dataset_name}_ROC_source.csv", index=False\n        )\n        pd.DataFrame(pr_rows).to_csv(\n            src_dir / f"{dataset_name}_PR_source.csv", index=False\n        )\n\n        # Per-class performance figure.\n        plot_pc = pc[pc["true_class_supported_in_domain"]].copy()\n        xx = np.arange(len(plot_pc))\n        ww = 0.20\n        fig, ax = plt.subplots(figsize=(11, 6))\n        ax.bar(xx - 1.5*ww, plot_pc["recall"], ww, label="Recall")\n        ax.bar(xx - 0.5*ww, plot_pc["f1"], ww, label="F1")\n        ax.bar(xx + 0.5*ww, plot_pc["roc_auc"], ww, label="ROC-AUC")\n        ax.bar(xx + 1.5*ww, plot_pc["average_precision"], ww, label="AP")\n        ax.set_xticks(xx, plot_pc["class_name"])\n        ax.set_ylim(0, 1.0)\n        ax.set_ylabel("Score")\n        ax.set_title(f"{dataset_name}: per-class performance")\n        ax.legend()\n        _save_figure(fig, fig_dir, f"{dataset_name}_per_class_performance")\n\n        # Confusion-pair error summary.\n        errs = []\n        for i, true_c in enumerate(CLASS_NAMES):\n            total_true = int(cm[i, :].sum())\n            for j, pred_c in enumerate(CLASS_NAMES):\n                if i == j or cm[i, j] == 0:\n                    continue\n                errs.append({\n                    "true_class": true_c,\n                    "predicted_class": pred_c,\n                    "count": int(cm[i, j]),\n                    "fraction_of_true_class": (\n                        float(cm[i, j] / total_true) if total_true else None\n                    ),\n                })\n        err_df = pd.DataFrame(\n            errs,\n            columns=["true_class", "predicted_class", "count", "fraction_of_true_class"]\n        )\n        if len(err_df):\n            err_df = err_df.sort_values(\n                ["count", "fraction_of_true_class"],\n                ascending=False\n            )\n        err_df.to_csv(tab_dir / f"{dataset_name}_error_pairs.csv", index=False)\n\n    # Suggested paper placement.\n    placement = {\n        "main_text": [\n            "tables/Table1_external_primary_metrics_with_95CI.csv",\n            "tables/Table2_internal_vs_external_equal_domain_mean.csv",\n            "figures/Fig1_external_domain_primary_metrics.png",\n            "figures/Fig2_internal_vs_external_mean.png",\n        ],\n        "supplement": [\n            "per-domain confusion matrices",\n            "per-domain ROC/PR curves",\n            "per-class metric tables",\n            "class-distribution tables",\n            "bootstrap details and raw predictions",\n        ],\n    }\n    (OUTPUT_ROOT / "paper_placement_suggestions.json").write_text(\n        json.dumps(placement, indent=2), encoding="utf-8"\n    )\n\n\ndef _build_material_index():\n    rows = []\n    for p in sorted(OUTPUT_ROOT.rglob("*")):\n        if not p.is_file():\n            continue\n        if "verified_checkpoints" in p.parts:\n            continue\n        rel = str(p.relative_to(OUTPUT_ROOT))\n        if rel.startswith("figures/"):\n            role = "publication_figure"\n        elif rel.startswith("tables/"):\n            role = "paper_table"\n        elif rel.startswith("figure_source_data/"):\n            role = "figure_source_data"\n        elif rel.startswith("model_predictions/") or rel.endswith("_predictions.csv"):\n            role = "prediction_source"\n        elif "bootstrap" in rel or rel.endswith("_metrics.json"):\n            role = "uncertainty_or_metric_source"\n        else:\n            role = "audit_or_supporting_material"\n        rows.append({\n            "file": rel,\n            "role": role,\n            "sha256": sha256_file(p),\n            "bytes": int(p.stat().st_size),\n        })\n    pd.DataFrame(rows).to_csv(OUTPUT_ROOT / "paper_material_index.csv", index=False)\n    (OUTPUT_ROOT / "paper_material_index.json").write_text(\n        json.dumps(rows, indent=2), encoding="utf-8"\n    )\n\n\ndef aggregate(bootstrap_iterations: int):\n    FINAL_ROOT.mkdir(parents=True, exist_ok=True)\n    manifests = decode_manifests()\n    _manifest_hash_guard(manifests)\n    frames = {name: resolve_manifest_paths(name, df) for name, df in manifests.items()}\n\n    summary_rows = []\n    all_results = {}\n    final_probs_by_domain = {}\n\n    for dataset_name, frame in frames.items():\n        model_probs = {}\n        key_cols = ["image_id_norm", "mapped_class", "group_id", "label"]\n\n        for model_name in ALL_MODEL_NAMES:\n            path = PRED_ROOT / model_name / f"{dataset_name}_tta7.csv"\n            if not path.exists():\n                raise FileNotFoundError(\n                    f"Missing model predictions for {dataset_name}: {path}"\n                )\n            df = pd.read_csv(path)\n            if model_name == ALL_MODEL_NAMES[0]:\n                ref = df[key_cols].copy()\n            elif not ref.equals(df[key_cols]):\n                raise RuntimeError(\n                    f"Model prediction row mismatch: {dataset_name} / {model_name}"\n                )\n            pcols = [f"prob_{c}" for c in CLASS_NAMES]\n            model_probs[model_name] = df[pcols].to_numpy(float)\n\n        b0_family = np.mean([model_probs[n] for n in B0_MODEL_NAMES], axis=0)\n        b3_family = np.mean([model_probs[n] for n in B3_MODEL_NAMES], axis=0)\n        ensemble = 0.5 * b0_family + 0.5 * b3_family\n\n        if not np.isfinite(ensemble).all():\n            raise RuntimeError(f"Non-finite final ensemble: {dataset_name}")\n        if not np.allclose(ensemble.sum(axis=1), 1.0, atol=2e-5, rtol=0):\n            raise RuntimeError(f"Final ensemble probabilities invalid: {dataset_name}")\n\n        final_probs_by_domain[dataset_name] = ensemble\n        y = frame["label"].to_numpy(int)\n        supported = tuple(DATASET_CONFIG[dataset_name]["supported_classes"])\n\n        metrics = metric_bundle(y, ensemble, supported)\n        ci = bootstrap_ci(\n            frame,\n            ensemble,\n            supported,\n            bootstrap_iterations,\n            DATASET_CONFIG[dataset_name]["bootstrap_seed"],\n        )\n\n        # Save family averages for exact reproducibility.\n        for fam_name, fam_probs in (("B0", b0_family), ("B3", b3_family)):\n            fam = frame[\n                ["image_id_norm", "image_hint", "mapped_class", "group_id", "image_path", "label"]\n            ].copy()\n            fam["family"] = fam_name\n            fam["predicted_label"] = fam_probs.argmax(axis=1)\n            fam["predicted_class"] = [\n                CLASS_NAMES[i] for i in fam["predicted_label"]\n            ]\n            for i, c in enumerate(CLASS_NAMES):\n                fam[f"prob_{c}"] = fam_probs[:, i]\n            fam.to_csv(\n                FINAL_ROOT / f"{dataset_name}_{fam_name.lower()}_family_equal3_tta7.csv",\n                index=False,\n            )\n\n        pred = frame[\n            ["image_id_norm", "image_hint", "mapped_class", "group_id", "image_path", "label"]\n        ].copy()\n        pred["predicted_label"] = ensemble.argmax(axis=1)\n        pred["predicted_class"] = [\n            CLASS_NAMES[i] for i in pred["predicted_label"]\n        ]\n        pred["correct"] = (pred["predicted_label"].to_numpy(int) == y).astype(int)\n        pred["max_probability"] = ensemble.max(axis=1)\n        clipped = np.clip(ensemble, 1e-12, 1.0)\n        pred["prediction_entropy"] = -(clipped * np.log(clipped)).sum(axis=1)\n        for i, c in enumerate(CLASS_NAMES):\n            pred[f"prob_{c}"] = ensemble[:, i]\n        pred.to_csv(\n            FINAL_ROOT / f"{dataset_name}_paperexp05a_predictions.csv",\n            index=False,\n        )\n\n        cm = pd.DataFrame(\n            np.asarray(metrics["confusion_matrix_7x7"], dtype=int),\n            index=[f"true_{c}" for c in CLASS_NAMES],\n            columns=[f"pred_{c}" for c in CLASS_NAMES],\n        )\n        cm.to_csv(FINAL_ROOT / f"{dataset_name}_confusion_matrix_7x7.csv")\n\n        dataset_result = {\n            "dataset": dataset_name,\n            "supported_true_classes": list(supported),\n            "model_output_classes": list(CLASS_NAMES),\n            "system": (\n                "fixed PAPER-EXP-01C source model: "\n                "equal 1/3 B0-300 TTA7 family + equal 1/3 B3-300 TTA7 family; "\n                "fixed 50/50 cross-family probability fusion"\n            ),\n            "metrics": metrics,\n            "bootstrap_ci": ci,\n        }\n        all_results[dataset_name] = dataset_result\n        with open(FINAL_ROOT / f"{dataset_name}_metrics.json", "w") as f:\n            json.dump(dataset_result, f, indent=2)\n\n        row = {\n            "dataset": dataset_name,\n            "n": metrics["n"],\n            "groups": int(frame["group_id"].nunique()),\n            "supported_true_classes": "|".join(supported),\n            "accuracy": metrics["accuracy"],\n            "macro_f1": metrics["macro_f1_supported_classes"],\n            "balanced_accuracy": metrics["balanced_accuracy_supported_classes"],\n            "melanoma_recall": metrics["melanoma_recall"],\n            "macro_auc": metrics["macro_auc_supported_classes_ovr"],\n            "predicted_DF_rate": metrics["predicted_DF_rate"],\n            "predicted_VASC_rate": metrics["predicted_VASC_rate"],\n        }\n        for mname, short in (\n            ("accuracy", "accuracy"),\n            ("macro_f1_supported_classes", "macro_f1"),\n            ("balanced_accuracy_supported_classes", "balanced_accuracy"),\n            ("melanoma_recall", "melanoma_recall"),\n            ("macro_auc_supported_classes_ovr", "macro_auc"),\n        ):\n            item = ci["metrics"][mname]\n            row[f"{short}_ci95_low"] = item["ci95_low"]\n            row[f"{short}_ci95_high"] = item["ci95_high"]\n            row[f"{short}_bootstrap_valid_n"] = item["valid_iterations"]\n        summary_rows.append(row)\n\n    summary_df = pd.DataFrame(summary_rows)\n    summary_df.to_csv(FINAL_ROOT / "external_summary.csv", index=False)\n\n    equal_domain_means = {\n        "accuracy": float(summary_df["accuracy"].mean()),\n        "macro_f1": float(summary_df["macro_f1"].mean()),\n        "balanced_accuracy": float(summary_df["balanced_accuracy"].mean()),\n        "melanoma_recall": float(summary_df["melanoma_recall"].mean()),\n        "macro_auc": float(summary_df["macro_auc"].mean()),\n    }\n\n    comparison = summary_df[\n        ["dataset", "accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"]\n    ].copy()\n    for metric in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"):\n        comparison[f"clean_internal_{metric}"] = CLEAN_INTERNAL_REFERENCE[metric]\n        comparison[f"{metric}_change_vs_clean_internal"] = (\n            comparison[metric] - CLEAN_INTERNAL_REFERENCE[metric]\n        )\n    comparison.to_csv(FINAL_ROOT / "external_vs_clean_internal.csv", index=False)\n\n    overall = {\n        "experiment": "PAPER-EXP-05A",\n        "code_version": CODE_VERSION,\n        "scientific_status": (\n            "frozen repeated external benchmark evaluation on previously audited/observed cohorts; "\n            "zero-shot with respect to the finalized PAPER-EXP-01C model"\n        ),\n        "clean_internal_reference": CLEAN_INTERNAL_REFERENCE,\n        "source_recovery_sha256": SOURCE_RECOVERY_SHA256,\n        "model": {\n            "class_order": list(CLASS_NAMES),\n            "input_size": IMAGE_SIZE,\n            "tta": list(TTA_VIEWS),\n            "B0_family": {\n                "checkpoint_sha256": {n: CHECKPOINT_HASHES[n] for n in B0_MODEL_NAMES},\n                "within_family_rule": "equal 1/3 probability average",\n            },\n            "B3_family": {\n                "checkpoint_sha256": {n: CHECKPOINT_HASHES[n] for n in B3_MODEL_NAMES},\n                "within_family_rule": "equal 1/3 probability average",\n            },\n            "cross_family_rule": "fixed 0.50 B0 family + 0.50 B3 family",\n            "threshold": "argmax",\n            "calibration": "none",\n        },\n        "external_manifest_sha256": EXPECTED_EXTERNAL_MANIFEST_SHA256,\n        "equal_domain_external_means": equal_domain_means,\n        "results": all_results,\n        "rules": [\n            "No training or adaptation on any external cohort.",\n            "No checkpoint selection from external labels.",\n            "No TTA selection from external labels.",\n            "No ensemble-weight search.",\n            "No threshold search.",\n            "No calibration.",\n            "No probability renormalization to externally supported classes.",\n            "DF/VASC source-head predictions count as ordinary errors.",\n            "External cohorts were historically observed in this wider project and are not claimed to be project-level pristine never-seen data.",\n            "PAPER-EXP-01C remains permanently frozen irrespective of these external results.",\n        ],\n        "decision": "REPORT_ALL_FOUR_EXTERNAL_DOMAINS_EXACTLY_AS_OBSERVED_NO_POST_EXTERNAL_TUNING",\n    }\n    with open(FINAL_ROOT / "all_external_results.json", "w") as f:\n        json.dump(overall, f, indent=2)\n\n    _publication_package(frames, final_probs_by_domain, summary_df)\n\n    manuscript_lines = [\n        "PAPER-EXP-05A — FROZEN EXTERNAL GENERALIZATION OF PAPER-EXP-01C",\n        "",\n        "Scientific status:",\n        "Frozen repeated external benchmark evaluation. These four cohorts were previously audited",\n        "and observed elsewhere in the project, so they are not described as pristine never-seen",\n        "project-level external tests. The finalized PAPER-EXP-01C model itself is unchanged.",\n        "",\n        "Exact source system:",\n        "3 x EfficientNet-B0 @ 300px + 3 x EfficientNet-B3 @ 300px;",\n        "TTA7 per model; equal 1/3 averaging within each family; fixed 50/50 family fusion.",\n        "",\n        "External results:",\n    ]\n    for _, row in summary_df.iterrows():\n        manuscript_lines.append(\n            f"{row[\'dataset\']}: n={int(row[\'n\'])}; "\n            f"accuracy={row[\'accuracy\']:.6f}; Macro-F1={row[\'macro_f1\']:.6f}; "\n            f"balanced accuracy={row[\'balanced_accuracy\']:.6f}; "\n            f"MEL recall={row[\'melanoma_recall\']:.6f}; Macro AUC={row[\'macro_auc\']:.6f}."\n        )\n    manuscript_lines += [\n        "",\n        "Equal-domain means:",\n        json.dumps(equal_domain_means, indent=2),\n        "",\n        "No post-external tuning, adaptation, threshold search, calibration or ensemble-weight change is allowed.",\n    ]\n    (OUTPUT_ROOT / "manuscript_result_block.txt").write_text(\n        "\\n".join(manuscript_lines), encoding="utf-8"\n    )\n\n    # Preserve exact code/protocol snapshot.\n    protocol_snapshot = {\'experiment\': \'PAPER-EXP-05A\', \'title\': \'Frozen External Generalization of Final PAPER-EXP-01C Model\', \'code_version\': \'PAPEREXP05A_FROZEN_PAPEREXP01C_ALL_EXTERNAL_2026-08-25_v2_NUMERICAL_FALLBACK\', \'status\': \'PREDECLARED_BEFORE_THIS_EXTERNAL RUN\', \'scientific_status\': \'Frozen repeated external benchmark evaluation. The four external cohorts were previously audited/observed in the wider project; they are not claimed as pristine never-seen project-level cohorts.\', \'final_internal_model\': {\'experiment\': \'PAPER-EXP-01C\', \'scientific_status\': \'final frozen lesion-safe clean internal benchmark\', \'test_images\': 971, \'test_lesions\': 749, \'test_manifest_sha256\': \'5c6f3cdca8ae60e200d6e5f5f1d619b3f08df71a2dadab95cdad7948a2834eec\', \'accuracy\': 0.8682, \'macro_f1\': 0.7333, \'balanced_accuracy\': 0.7076, \'melanoma_recall\': 0.7524, \'macro_auc\': 0.9756}, \'source_checkpoint_recovery\': {\'PAPEREXP01A_B0_Recovery_SHA256\': \'778060755a8be16d8ab7bb21055d50a6f0a641a763f19caf30da399096192668\', \'PAPEREXP01B_B3_Recovery_SHA256\': \'2a25768706d4bd7ee7ce6f6982ceedd6b0b97de936cf3739f68cdba7e2bc9ae8\'}, \'model\': {\'architecture\': \'3x EfficientNet-B0 + 3x EfficientNet-B3\', \'input_size\': 300, \'class_order\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\', \'DF\', \'VASC\'], \'B0_checkpoint_sha256\': {\'b0_seed_20260820\': \'30efcb4f88cfddd4f6ece227d920fbab27caf38e821f2e442941510b199bfb39\', \'b0_seed_20260821\': \'a79eef838354a6613d75b7aa455d643ccd0f3309b23a31f177c3169f723575cb\', \'b0_seed_20260822\': \'f258083d9bb36858ad96ce2d2fd8ba54a0834dc5b8f38dcbb1ee762783509587\'}, \'B3_checkpoint_sha256\': {\'b3_seed_20260820\': \'59a23d41c48cb1ccbf9184e9cf67c115717775560283bb4a1da95f14698dc296\', \'b3_seed_20260821\': \'723176d2c08b759b9a8c2cbbfe245840ad844f1a77da748948c75c09afc51d69\', \'b3_seed_20260822\': \'2dc2cf4d28b6f5b8ccdb61a173aff25ba80ab2e0c794dfdfafcb104a87d89a4d\'}, \'TTA7\': [\'original\', \'horizontal\', \'vertical\', \'horizontal_vertical\', \'rotation90\', \'rotation180\', \'rotation270\'], \'B0_family_rule\': \'equal 1/3 probability average\', \'B3_family_rule\': \'equal 1/3 probability average\', \'final_fusion\': \'fixed 0.50 B0 family + 0.50 B3 family\', \'threshold\': \'argmax\', \'calibration\': \'none\'}, \'external_cohorts\': {\'DERM12345\': {\'n\': 2321, \'groups\': 574, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\'], \'manifest_sha256\': \'9becd1ea91602dca071bb7f561011d0fc6f46266b86d148ac3b3d463db909453\'}, \'BCN20000\': {\'n\': 11747, \'groups\': 3377, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\'], \'manifest_sha256\': \'d9eeacfe22638dca39eb052ddad78c418f783f62f258551cb5767c734fa0debd\'}, \'Derm7pt\': {\'n\': 355, \'groups\': 355, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'BKL\'], \'manifest_sha256\': \'47e092f799d8c4c97cd94ccb3116995ad03e00b8e249f24b1edfdbcaf9dc5299\'}, \'PAD_UFES_20\': {\'n\': 2092, \'groups\': 1297, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\'], \'manifest_sha256\': \'18f07c4079735c533c1ab8ba17e77377f43143f052fcb4bde468cc255b453bef\'}}, \'metrics\': [\'accuracy\', \'Macro-F1 over supported true classes\', \'balanced accuracy over supported true classes\', \'melanoma recall\', \'macro one-vs-rest ROC-AUC\', \'per-class precision/recall/specificity/F1/AUC/AP\'], \'uncertainty\': {\'method\': \'group-level percentile bootstrap\', \'iterations\': 2000, \'confidence_level\': 0.95, \'group_column\': \'audit-frozen group_id\', \'rule\': \'resamples missing a supported true class are skipped\'}, \'hard_rules\': [\'No external training or adaptation.\', \'No checkpoint selection using external labels.\', \'No TTA search.\', \'No ensemble-weight search.\', \'No threshold search.\', \'No calibration.\', \'No renormalization to supported external classes.\', \'DF/VASC source-head predictions count as errors.\', \'Report all four domains regardless of outcome.\', \'PAPER-EXP-01C remains permanently frozen after this run.\']}\n    (OUTPUT_ROOT / "paper_exp05a_protocol.json").write_text(\n        json.dumps(protocol_snapshot, indent=2), encoding="utf-8"\n    )\n    try:\n        shutil.copy2(Path(__file__), OUTPUT_ROOT / "paper_exp05a_code_snapshot.py")\n    except Exception:\n        pass\n\n    _build_material_index()\n    make_recovery_zip()\n\n    result_zip = WORKING_ROOT / "PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Results.zip"\n    package_output(result_zip)\n\n    print("\\n" + "=" * 116)\n    print("PAPER-EXP-05A COMPLETE")\n    print("=" * 116)\n    print(summary_df.to_string(index=False))\n    print("\\nEQUAL-DOMAIN EXTERNAL MEANS:")\n    print(json.dumps(equal_domain_means, indent=2))\n    print("\\nPOST-EXTERNAL TUNING ALLOWED:", False)\n    print("\\nRESULTS ZIP:", result_zip)\n    print("SHA256:", sha256_file(result_zip))\n    print("RECOVERY ZIP:", WORKING_ROOT / "PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip")\n\n\ndef package_output(zip_path: Path):\n    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(OUTPUT_ROOT.rglob("*")):\n            if not p.is_file():\n                continue\n            if "verified_checkpoints" in p.parts:\n                continue\n            z.write(p, arcname=str(p.relative_to(OUTPUT_ROOT)))\n\n\ndef make_recovery_zip():\n    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n    recovery = WORKING_ROOT / "PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip"\n    with zipfile.ZipFile(recovery, "w", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(OUTPUT_ROOT.rglob("*")):\n            if not p.is_file():\n                continue\n            if "verified_checkpoints" in p.parts:\n                continue\n            z.write(p, arcname=str(p.relative_to(OUTPUT_ROOT)))\n    print("[RECOVERY] Updated:", recovery)\n\n\ndef preflight():\n    print("=" * 108)\n    print(CODE_VERSION)\n    print("PAPER-EXP-05A — FROZEN PAPER-EXP-01C ALL-EXTERNAL BENCHMARK")\n    print("=" * 108)\n    print("torch:", torch.__version__)\n    try:\n        import torchvision as _tv\n        print("torchvision:", _tv.__version__)\n    except Exception:\n        pass\n\n    print("CUDA available:", torch.cuda.is_available())\n    print("GPU count:", torch.cuda.device_count())\n    for i in range(torch.cuda.device_count()):\n        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU accelerator.")\n\n    manifests = decode_manifests()\n    _manifest_hash_guard(manifests)\n    for name, df in manifests.items():\n        print(\n            f"Frozen manifest {name}: n={len(df)}, "\n            f"sha256={df[\'manifest_sha256\'].iloc[0]}"\n        )\n\n    resolved_frames = {\n        name: resolve_manifest_paths(name, df)\n        for name, df in manifests.items()\n    }\n    for name, frame in resolved_frames.items():\n        counts = frame["mapped_class"].value_counts().to_dict()\n        print(\n            f"{name}: resolved n={len(frame)}, groups={frame[\'group_id\'].nunique()}, "\n            f"counts={counts}"\n        )\n\n    checkpoints = find_exact_checkpoints()\n    print("Exact frozen PAPER-EXP-01C source checkpoints:")\n    for model_name, path in checkpoints.items():\n        print(f"  {model_name}: {path} | {sha256_file(path)}")\n\n    for model_name, path in checkpoints.items():\n        family = _family_for_model(model_name)\n        m = load_model(path, torch.device("cpu"), family)\n        print(f"Strict CPU load PASS: {model_name} ({family})")\n        del m\n\n    preflight_summary = {\n        "experiment": "PAPER-EXP-05A",\n        "code_version": CODE_VERSION,\n        "scientific_status": (\n            "frozen repeated external benchmark; no external training/selection"\n        ),\n        "checkpoint_sha256": {\n            name: sha256_file(path)\n            for name, path in checkpoints.items()\n        },\n        "external_manifest_sha256": EXPECTED_EXTERNAL_MANIFEST_SHA256,\n        "cohorts": {\n            name: {\n                "n": int(len(frame)),\n                "groups": int(frame["group_id"].nunique()),\n                "counts": {\n                    str(k): int(v)\n                    for k, v in frame["mapped_class"].value_counts().to_dict().items()\n                },\n            }\n            for name, frame in resolved_frames.items()\n        },\n        "source_recovery_sha256": SOURCE_RECOVERY_SHA256,\n        "test_or_external_training": False,\n        "external_adaptation": False,\n        "ensemble_weight_search": False,\n        "tta_search": False,\n        "threshold_search": False,\n        "calibration": False,\n    }\n    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n    script_sha256 = sha256_file(Path(__file__).resolve())\n    preflight_summary["script_sha256"] = script_sha256\n    (OUTPUT_ROOT / "preflight_summary.json").write_text(\n        json.dumps(preflight_summary, indent=2), encoding="utf-8"\n    )\n\n    protocol_snapshot = {\'experiment\': \'PAPER-EXP-05A\', \'title\': \'Frozen External Generalization of Final PAPER-EXP-01C Model\', \'code_version\': \'PAPEREXP05A_FROZEN_PAPEREXP01C_ALL_EXTERNAL_2026-08-25_v2_NUMERICAL_FALLBACK\', \'status\': \'PREDECLARED_BEFORE_THIS_EXTERNAL RUN\', \'scientific_status\': \'Frozen repeated external benchmark evaluation. The four external cohorts were previously audited/observed in the wider project; they are not claimed as pristine never-seen project-level cohorts.\', \'final_internal_model\': {\'experiment\': \'PAPER-EXP-01C\', \'scientific_status\': \'final frozen lesion-safe clean internal benchmark\', \'test_images\': 971, \'test_lesions\': 749, \'test_manifest_sha256\': \'5c6f3cdca8ae60e200d6e5f5f1d619b3f08df71a2dadab95cdad7948a2834eec\', \'accuracy\': 0.8682, \'macro_f1\': 0.7333, \'balanced_accuracy\': 0.7076, \'melanoma_recall\': 0.7524, \'macro_auc\': 0.9756}, \'source_checkpoint_recovery\': {\'PAPEREXP01A_B0_Recovery_SHA256\': \'778060755a8be16d8ab7bb21055d50a6f0a641a763f19caf30da399096192668\', \'PAPEREXP01B_B3_Recovery_SHA256\': \'2a25768706d4bd7ee7ce6f6982ceedd6b0b97de936cf3739f68cdba7e2bc9ae8\'}, \'model\': {\'architecture\': \'3x EfficientNet-B0 + 3x EfficientNet-B3\', \'input_size\': 300, \'class_order\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\', \'DF\', \'VASC\'], \'B0_checkpoint_sha256\': {\'b0_seed_20260820\': \'30efcb4f88cfddd4f6ece227d920fbab27caf38e821f2e442941510b199bfb39\', \'b0_seed_20260821\': \'a79eef838354a6613d75b7aa455d643ccd0f3309b23a31f177c3169f723575cb\', \'b0_seed_20260822\': \'f258083d9bb36858ad96ce2d2fd8ba54a0834dc5b8f38dcbb1ee762783509587\'}, \'B3_checkpoint_sha256\': {\'b3_seed_20260820\': \'59a23d41c48cb1ccbf9184e9cf67c115717775560283bb4a1da95f14698dc296\', \'b3_seed_20260821\': \'723176d2c08b759b9a8c2cbbfe245840ad844f1a77da748948c75c09afc51d69\', \'b3_seed_20260822\': \'2dc2cf4d28b6f5b8ccdb61a173aff25ba80ab2e0c794dfdfafcb104a87d89a4d\'}, \'TTA7\': [\'original\', \'horizontal\', \'vertical\', \'horizontal_vertical\', \'rotation90\', \'rotation180\', \'rotation270\'], \'B0_family_rule\': \'equal 1/3 probability average\', \'B3_family_rule\': \'equal 1/3 probability average\', \'final_fusion\': \'fixed 0.50 B0 family + 0.50 B3 family\', \'threshold\': \'argmax\', \'calibration\': \'none\'}, \'external_cohorts\': {\'DERM12345\': {\'n\': 2321, \'groups\': 574, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\'], \'manifest_sha256\': \'9becd1ea91602dca071bb7f561011d0fc6f46266b86d148ac3b3d463db909453\'}, \'BCN20000\': {\'n\': 11747, \'groups\': 3377, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\'], \'manifest_sha256\': \'d9eeacfe22638dca39eb052ddad78c418f783f62f258551cb5767c734fa0debd\'}, \'Derm7pt\': {\'n\': 355, \'groups\': 355, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'BKL\'], \'manifest_sha256\': \'47e092f799d8c4c97cd94ccb3116995ad03e00b8e249f24b1edfdbcaf9dc5299\'}, \'PAD_UFES_20\': {\'n\': 2092, \'groups\': 1297, \'supported_true_classes\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\'], \'manifest_sha256\': \'18f07c4079735c533c1ab8ba17e77377f43143f052fcb4bde468cc255b453bef\'}}, \'metrics\': [\'accuracy\', \'Macro-F1 over supported true classes\', \'balanced accuracy over supported true classes\', \'melanoma recall\', \'macro one-vs-rest ROC-AUC\', \'per-class precision/recall/specificity/F1/AUC/AP\'], \'uncertainty\': {\'method\': \'group-level percentile bootstrap\', \'iterations\': 2000, \'confidence_level\': 0.95, \'group_column\': \'audit-frozen group_id\', \'rule\': \'resamples missing a supported true class are skipped\'}, \'hard_rules\': [\'No external training or adaptation.\', \'No checkpoint selection using external labels.\', \'No TTA search.\', \'No ensemble-weight search.\', \'No threshold search.\', \'No calibration.\', \'No renormalization to supported external classes.\', \'DF/VASC source-head predictions count as errors.\', \'Report all four domains regardless of outcome.\', \'PAPER-EXP-01C remains permanently frozen after this run.\']}\n    protocol_snapshot["script_sha256_at_runtime"] = script_sha256\n    (OUTPUT_ROOT / "protocol_snapshot.json").write_text(\n        json.dumps(protocol_snapshot, indent=2), encoding="utf-8"\n    )\n    print("Script SHA256:", script_sha256)\n    print("\\nPRE-FLIGHT PASS")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--mode",\n        choices=("all", "preflight", "predict", "aggregate"),\n        default="all",\n    )\n    parser.add_argument("--workers", type=int, default=4)\n    parser.add_argument("--bootstrap", type=int, default=2000)\n    args = parser.parse_args()\n\n    if args.mode == "preflight":\n        preflight()\n    elif args.mode == "predict":\n        run_models(args.workers)\n    elif args.mode == "aggregate":\n        aggregate(args.bootstrap)\n    else:\n        preflight()\n        run_models(args.workers)\n        aggregate(args.bootstrap)\n\n\nif __name__ == "__main__":\n    main()\n'
p = Path("/kaggle/working/PAPEREXP05A_frozen_paperexp01c_all_external_v2_NUMERICAL_FALLBACK.py")
p.write_text(SCRIPT, encoding="utf-8")
print("Wrote:", p)
print("Code version:", 'PAPEREXP05A_FROZEN_PAPEREXP01C_ALL_EXTERNAL_2026-08-25_v2_NUMERICAL_FALLBACK')
print("Script bytes:", p.stat().st_size)


Wrote: /kaggle/working/PAPEREXP05A_frozen_paperexp01c_all_external_v2_NUMERICAL_FALLBACK.py
Code version: PAPEREXP05A_FROZEN_PAPEREXP01C_ALL_EXTERNAL_2026-08-25_v2_NUMERICAL_FALLBACK
Script bytes: 293450


In [2]:
import subprocess, sys

cmd = [
    sys.executable,
    "-u",
    "/kaggle/working/PAPEREXP05A_frozen_paperexp01c_all_external_v2_NUMERICAL_FALLBACK.py",
    "--mode", "all",
    "--workers", "4",
    "--bootstrap", "2000",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /usr/bin/python3 -u /kaggle/working/PAPEREXP05A_frozen_paperexp01c_all_external_v2_NUMERICAL_FALLBACK.py --mode all --workers 4 --bootstrap 2000
PAPEREXP05A_FROZEN_PAPEREXP01C_ALL_EXTERNAL_2026-08-25_v2_NUMERICAL_FALLBACK
PAPER-EXP-05A — FROZEN PAPER-EXP-01C ALL-EXTERNAL BENCHMARK
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Frozen manifest DERM12345: n=2321, sha256=9becd1ea91602dca071bb7f561011d0fc6f46266b86d148ac3b3d463db909453
Frozen manifest BCN20000: n=11747, sha256=d9eeacfe22638dca39eb052ddad78c418f783f62f258551cb5767c734fa0debd
Frozen manifest Derm7pt: n=355, sha256=47e092f799d8c4c97cd94ccb3116995ad03e00b8e249f24b1edfdbcaf9dc5299
Frozen manifest PAD_UFES_20: n=2092, sha256=18f07c4079735c533c1ab8ba17e77377f43143f052fcb4bde468cc255b453bef
[DATASET] DERM12345: indexed 12345 image files
[DATASET] BCN20000: indexed 18946 image files
[DATASET] Derm7pt: indexed 2013 image files
[DATASET] PAD_UFES_20: indexe

b0_seed_20260820 | DERM12345 | fixed TTA7: 100%|██████████| 37/37 [00:53<00:00,  1.45s/it]
b0_seed_20260820 | BCN20000 | fixed TTA7:   0%|          | 0/184 [00:00<?, ?it/s]

[SAVED] /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal/model_predictions/b0_seed_20260820/DERM12345_tta7.csv
[RECOVERY] Updated: /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip


b0_seed_20260820 | BCN20000 | fixed TTA7: 100%|██████████| 184/184 [02:03<00:00,  1.49it/s]


[SAVED] /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal/model_predictions/b0_seed_20260820/BCN20000_tta7.csv
[RECOVERY] Updated: /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip


...

[SAVED] /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal/model_predictions/b3_seed_20260822/Derm7pt_tta7.csv
[RECOVERY] Updated: /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip


b3_seed_20260822 | PAD_UFES_20 | fixed TTA7: 100%|██████████| 53/53 [00:54<00:00,  1.02s/it]


[SAVED] /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal/model_predictions/b3_seed_20260822/PAD_UFES_20_tta7.csv
[RECOVERY] Updated: /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip
[DATASET] DERM12345: indexed 12345 image files
[DATASET] BCN20000: indexed 18946 image files
[DATASET] Derm7pt: indexed 2013 image files
[DATASET] PAD_UFES_20: indexed 2298 image files


[RECOVERY] Updated: /kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip

PAPER-EXP-05A COMPLETE
    dataset     n  groups supported_true_classes  accuracy  macro_f1  balanced_accuracy  melanoma_recall  macro_auc  predicted_DF_rate  predicted_VASC_rate  accuracy_ci95_low  accuracy_ci95_high  accuracy_bootstrap_valid_n  macro_f1_ci95_low  macro_f1_ci95_high  macro_f1_bootstrap_valid_n  balanced_accuracy_ci95_low  balanced_accuracy_ci95_high  balanced_accuracy_bootstrap_valid_n  melanoma_recall_ci95_low  melanoma_recall_ci95_high  melanoma_recall_bootstrap_valid_n  macro_auc_ci95_low  macro_auc_ci95_high  macro_auc_bootstrap_valid_n
  DERM12345  2321     574   MEL|NV|BCC|AKIEC|BKL  0.898320  0.532730           0.474100         0.548780   0.933170           0.002585             0.000431           0.873273            0.920542                        2000           0.440788            0.606703                        2000                    0.402295                     0.54

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/working/PAPEREXP05A_frozen_paperexp01c_all_external_v2_NUMERICAL_FALLBACK.py', '--mode', 'all', '--workers', '4', '--bootstrap', '2000'], returncode=0)

In [3]:
from pathlib import Path
import hashlib, json

for name in [
    "PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Results.zip",
    "PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip",
]:
    p = Path("/kaggle/working") / name
    print("\nUPLOAD THIS FILE:")
    print(p)
    print("Exists:", p.exists())
    if p.exists():
        print("Size:", p.stat().st_size)
        print("SHA256:", hashlib.sha256(p.read_bytes()).hexdigest())

log = Path("/kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal/numerical_precision_fallback_log.json")
print("\nNumerical fallback log exists:", log.exists())
if log.exists():
    print(log.read_text())



UPLOAD THIS FILE:
/kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Results.zip
Exists: True
Size: 13383016
SHA256: 377dad6e17e1d5f0603a953579b2227afde3033965a027e5309c3e1469c1b5e4

UPLOAD THIS FILE:
/kaggle/working/PAPEREXP05A_Frozen_PAPEREXP01C_AllExternal_Recovery.zip
Exists: True
Size: 13383016
SHA256: 377dad6e17e1d5f0603a953579b2227afde3033965a027e5309c3e1469c1b5e4

Numerical fallback log exists: True
[
  {
    "description": "b3_seed_20260821 | PAD_UFES_20 | fixed TTA7",
    "row_indices": [
      811
    ],
    "n_rows": 1,
    "primary_precision": "CUDA autocast float16",
    "fallback_precision": "float32",
    "model_changed": false,
    "tta_changed": false,
    "weights_changed": false,
    "threshold_changed": false,
    "calibration_changed": false,
    "labels_used_for_fallback": false
  },
  {
    "description": "b3_seed_20260821 | PAD_UFES_20 | fixed TTA7",
    "row_indices": [
      1857
    ],
    "n_rows": 1,
    "primary_precision": "CUDA autocast float16